# Explore referral / mental-health-service signals (no ICD F20 required)

**Question from Yize**: are there patients in YNHH who were referred to specialty mental
health care without ever receiving a schizophrenia/psychosis ICD code? If yes, we might
want to broaden the case definition to include them, not just F20-coded patients.

OMOP CDM places referral-like signals across several tables, so we scan all plausible
locations:

| where | what to look for |
|---|---|
| `visit_occurrence` | visits with `visit_source_value` / `visit_concept_id` indicating psych/behavioral health |
| `care_site` | care sites named like psychiatry, behavioral health, mental health |
| `procedure_occurrence` | procedure codes for referral, consultation, psychotherapy, etc. |
| `observation` | "referral to" observations |
| `condition_occurrence` | broader psychiatric ICDs (F2x, F3x, F4x) outside F20 |
| `note_nlp` (if usable) | NLP-extracted "refer to psychiatry" mentions |

This notebook produces summary counts at each layer and outputs a candidate person_id
list of people who look like they had specialty mental health contact but no F20 code.


In [ ]:
import os
from pathlib import Path
import pandas as pd
import duckdb

pd.set_option('display.max_columns', 120)
pd.set_option('display.max_rows', 200)
pd.set_option('display.width', 200)

path = '/home/jupyter/2836994-data2'
OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

con = duckdb.connect()
print('DuckDB:', duckdb.__version__)
print('Data path:', path)


## 1. List what tables exist + sample one column to understand the data

In [ ]:
# Confirm which tables are present at the data path
present = sorted([p.name for p in Path(path).iterdir() if p.is_dir()])
print('Tables present:')
for t in present:
    print('  ', t)


## 2. Get a reference set: persons who DO have any F20.* code

This is our anchor. Anyone in the referral-signal sets but NOT in this set is a
candidate "psych-care-without-F20-diagnosis" patient.


In [ ]:
f20_persons = con.sql(f"""
SELECT DISTINCT person_id
FROM read_parquet('{path}/condition_occurrence/*.parquet')
WHERE condition_source_value LIKE 'F20%'
""").to_df()

f20_set = set(f20_persons['person_id'].tolist())
print(f'Persons with any F20 code: {len(f20_set):,}')


## 3. Scan visit_occurrence for psychiatric / behavioral-health visits

`visit_source_value` often carries free-text-ish strings like "Psychiatry Outpatient",
"Behavioral Health Inpatient", "Mental Health" etc. We look for those substrings.


In [ ]:
# First: peek at what visit_source_value values look like and how frequent they are
visit_src_dist = con.sql(f"""
SELECT
  visit_source_value,
  COUNT(*) AS n_rows,
  COUNT(DISTINCT person_id) AS n_persons
FROM read_parquet('{path}/visit_occurrence/*.parquet')
WHERE visit_source_value IS NOT NULL
GROUP BY visit_source_value
ORDER BY n_persons DESC
LIMIT 40
""").to_df()
print('Top 40 visit_source_value strings:')
display(visit_src_dist)


In [ ]:
# Now: filter to visits whose source_value mentions psych/behavioral/mental
visit_psych = con.sql(f"""
SELECT
  visit_source_value,
  COUNT(*) AS n_rows,
  COUNT(DISTINCT person_id) AS n_persons
FROM read_parquet('{path}/visit_occurrence/*.parquet')
WHERE visit_source_value IS NOT NULL
  AND (
       lower(visit_source_value) LIKE '%psych%'
    OR lower(visit_source_value) LIKE '%behavioral%'
    OR lower(visit_source_value) LIKE '%mental%'
    OR lower(visit_source_value) LIKE '%behaviour%'
  )
GROUP BY visit_source_value
ORDER BY n_persons DESC
""").to_df()
print('Psych/behavioral-flagged visit_source_value strings:')
display(visit_psych)
print(f'Total distinct visit_source_value strings matched: {len(visit_psych)}')

# Collect the matching person_ids
visit_psych_persons = con.sql(f"""
SELECT DISTINCT person_id
FROM read_parquet('{path}/visit_occurrence/*.parquet')
WHERE visit_source_value IS NOT NULL
  AND (
       lower(visit_source_value) LIKE '%psych%'
    OR lower(visit_source_value) LIKE '%behavioral%'
    OR lower(visit_source_value) LIKE '%mental%'
    OR lower(visit_source_value) LIKE '%behaviour%'
  )
""").to_df()
print(f'\nPersons with at least one psych/behavioral visit (by visit_source_value): {len(visit_psych_persons):,}')


## 4. Cross-reference visit_occurrence -> care_site -> care_site_name

`visit_occurrence.care_site_id` joins to `care_site`, which carries the department name.
This catches psych care delivered without it being in the visit_source_value text.


In [ ]:
# Peek at care_site columns
care_site_cols = con.sql(f"""
DESCRIBE SELECT * FROM read_parquet('{path}/care_site/*.parquet') LIMIT 1
""").to_df()
print('care_site columns:')
display(care_site_cols)


In [ ]:
# Find care_sites named like psychiatry/behavioral/mental health
psych_care_sites = con.sql(f"""
SELECT
  care_site_id,
  care_site_name,
  care_site_source_value,
  place_of_service_source_value
FROM read_parquet('{path}/care_site/*.parquet')
WHERE
     lower(coalesce(care_site_name, '')) LIKE '%psych%'
  OR lower(coalesce(care_site_name, '')) LIKE '%behavioral%'
  OR lower(coalesce(care_site_name, '')) LIKE '%mental%'
  OR lower(coalesce(care_site_source_value, '')) LIKE '%psych%'
  OR lower(coalesce(care_site_source_value, '')) LIKE '%behavioral%'
  OR lower(coalesce(care_site_source_value, '')) LIKE '%mental%'
ORDER BY care_site_name
""").to_df()
print(f'Psychiatric/behavioral care sites identified: {len(psych_care_sites)}')
display(psych_care_sites.head(40))


In [ ]:
# Persons with at least one visit at a psych care site
if len(psych_care_sites) > 0:
    psych_site_ids = psych_care_sites['care_site_id'].dropna().astype('int64').tolist()
    psych_site_id_sql = ','.join(str(x) for x in psych_site_ids)

    care_site_persons = con.sql(f"""
    SELECT DISTINCT person_id
    FROM read_parquet('{path}/visit_occurrence/*.parquet')
    WHERE care_site_id IN ({psych_site_id_sql})
    """).to_df()
    print(f'Persons with >=1 visit at a psych care site: {len(care_site_persons):,}')
else:
    care_site_persons = pd.DataFrame({'person_id': []})
    print('No psych care sites identified; skipping care_site_id-based person lookup.')


## 5. Procedure_occurrence: psychotherapy / psych consultation / behavioral health CPT codes

Common CPT codes for behavioral health:
- 90791, 90792  Psychiatric diagnostic evaluation
- 90832-90838   Psychotherapy
- 90839, 90840  Psychotherapy for crisis
- 90845-90853   Group / family psychotherapy
- 90875, 90876  Biofeedback with psychotherapy
- 99492-99494   Behavioral health care management
- H0001-H2037   HCPCS behavioral health codes


In [ ]:
# Pull procedures with relevant CPT/HCPCS or with text mentions of psych/behavioral
psych_proc_patterns_sql = """
   regexp_matches(coalesce(procedure_source_value, ''), '^9079[12]$')
OR regexp_matches(coalesce(procedure_source_value, ''), '^9083[2-9]$')
OR regexp_matches(coalesce(procedure_source_value, ''), '^9084[0-9]$')
OR regexp_matches(coalesce(procedure_source_value, ''), '^9085[0-9]$')
OR regexp_matches(coalesce(procedure_source_value, ''), '^9087[56]$')
OR regexp_matches(coalesce(procedure_source_value, ''), '^9949[234]$')
OR regexp_matches(coalesce(procedure_source_value, ''), '^H[0-9]{4}$')
OR lower(coalesce(procedure_source_value, '')) LIKE '%psych%'
OR lower(coalesce(procedure_source_value, '')) LIKE '%behavioral%'
OR lower(coalesce(procedure_source_value, '')) LIKE '%mental%'
"""

# Top procedure_source_value strings that match (sanity)
psych_proc_codes = con.sql(f"""
SELECT
  procedure_source_value,
  COUNT(*) AS n_rows,
  COUNT(DISTINCT person_id) AS n_persons
FROM read_parquet('{path}/procedure_occurrence/*.parquet')
WHERE {psych_proc_patterns_sql}
GROUP BY procedure_source_value
ORDER BY n_persons DESC
LIMIT 50
""").to_df()
print('Top psych/behavioral procedure codes (procedure_source_value):')
display(psych_proc_codes)
print(f'Distinct matching codes shown above: {len(psych_proc_codes)}')


In [ ]:
# Persons with any matching psych procedure
proc_persons = con.sql(f"""
SELECT DISTINCT person_id
FROM read_parquet('{path}/procedure_occurrence/*.parquet')
WHERE {psych_proc_patterns_sql}
""").to_df()
print(f'Persons with >=1 psych/behavioral procedure code: {len(proc_persons):,}')


## 6. Observation table: referral / mental health observations

In [ ]:
# Peek at observation columns to know what's available
obs_cols = con.sql(f"""
DESCRIBE SELECT * FROM read_parquet('{path}/observation/*.parquet') LIMIT 1
""").to_df()
print('observation columns:')
display(obs_cols)


In [ ]:
# Look for observations whose source_value mentions referral OR psych
# Some EHRs put 'Referral to Psychiatry' / 'Mental Health Referral' here as a free-text observation
obs_psych = con.sql(f"""
SELECT
  observation_source_value,
  COUNT(*) AS n_rows,
  COUNT(DISTINCT person_id) AS n_persons
FROM read_parquet('{path}/observation/*.parquet')
WHERE observation_source_value IS NOT NULL
  AND (
       (lower(observation_source_value) LIKE '%refer%'
         AND (lower(observation_source_value) LIKE '%psych%'
              OR lower(observation_source_value) LIKE '%behavioral%'
              OR lower(observation_source_value) LIKE '%mental%'))
    OR lower(observation_source_value) LIKE '%psychiatry%'
    OR lower(observation_source_value) LIKE '%behavioral health%'
    OR lower(observation_source_value) LIKE '%mental health%'
  )
GROUP BY observation_source_value
ORDER BY n_persons DESC
LIMIT 50
""").to_df()
print('Top observation_source_value strings indicating psych/referral:')
display(obs_psych)

obs_persons = con.sql(f"""
SELECT DISTINCT person_id
FROM read_parquet('{path}/observation/*.parquet')
WHERE observation_source_value IS NOT NULL
  AND (
       (lower(observation_source_value) LIKE '%refer%'
         AND (lower(observation_source_value) LIKE '%psych%'
              OR lower(observation_source_value) LIKE '%behavioral%'
              OR lower(observation_source_value) LIKE '%mental%'))
    OR lower(observation_source_value) LIKE '%psychiatry%'
    OR lower(observation_source_value) LIKE '%behavioral health%'
    OR lower(observation_source_value) LIKE '%mental health%'
  )
""").to_df()
print(f'\nPersons with a referral/psych-related observation: {len(obs_persons):,}')


## 7. condition_occurrence: broader psychiatric ICDs OUTSIDE F20

Some patients may have psychosis-related diagnoses other than F20, which indicate
psychiatric care contact:
- F22  Delusional disorders
- F23  Brief psychotic disorder
- F24  Shared psychotic disorder
- F25  Schizoaffective disorder
- F28  Other psychotic disorder
- F29  Unspecified psychosis
- F30-F31  Bipolar (often co-treated by same services)
- F32-F33  Depression (huge n; only include if also relevant per Yize)

We pull F2x first, since those are closest to schizophrenia in service-need.


In [ ]:
psychosis_other_persons = con.sql(f"""
SELECT DISTINCT person_id
FROM read_parquet('{path}/condition_occurrence/*.parquet')
WHERE condition_source_value LIKE 'F22%'
   OR condition_source_value LIKE 'F23%'
   OR condition_source_value LIKE 'F24%'
   OR condition_source_value LIKE 'F25%'
   OR condition_source_value LIKE 'F28%'
   OR condition_source_value LIKE 'F29%'
""").to_df()
print(f'Persons with F22-F29 (non-F20 psychosis spectrum): {len(psychosis_other_persons):,}')

# Breakdown by ICD prefix for context
psychosis_breakdown = con.sql(f"""
SELECT
  substr(condition_source_value, 1, 3) AS icd_prefix,
  COUNT(*) AS n_rows,
  COUNT(DISTINCT person_id) AS n_persons
FROM read_parquet('{path}/condition_occurrence/*.parquet')
WHERE condition_source_value LIKE 'F22%' OR condition_source_value LIKE 'F23%'
   OR condition_source_value LIKE 'F24%' OR condition_source_value LIKE 'F25%'
   OR condition_source_value LIKE 'F28%' OR condition_source_value LIKE 'F29%'
GROUP BY icd_prefix
ORDER BY n_persons DESC
""").to_df()
display(psychosis_breakdown)


## 8. Union all referral signals and compute overlap with F20

In [ ]:
signal_sets = {
    'F20 ICD (anchor)':              f20_set,
    'visit_source_value psych':      set(visit_psych_persons['person_id'].tolist()),
    'care_site psych':               set(care_site_persons['person_id'].tolist()),
    'procedure psych':               set(proc_persons['person_id'].tolist()),
    'observation referral/psych':    set(obs_persons['person_id'].tolist()),
    'non-F20 psychosis (F22-F29)':   set(psychosis_other_persons['person_id'].tolist()),
}

print('Signal source person counts:')
for name, s in signal_sets.items():
    print(f'  {name:32s}  n_persons = {len(s):>10,}')

# Union of ALL non-F20 signals
non_f20_signals = (signal_sets['visit_source_value psych']
                   | signal_sets['care_site psych']
                   | signal_sets['procedure psych']
                   | signal_sets['observation referral/psych']
                   | signal_sets['non-F20 psychosis (F22-F29)'])

print()
print(f'Union of all non-F20 signals: {len(non_f20_signals):,}')
print(f'  of which ALSO have F20:     {len(non_f20_signals & f20_set):,}')
print(f'  of which have NO F20:       {len(non_f20_signals - f20_set):,}  <-- candidate broadened cohort')


## 9. Decompose: how many "no F20 but has signal" come from each source?

This tells you which signal is doing the work of expanding the cohort.


In [ ]:
no_f20 = non_f20_signals - f20_set

source_contribution = []
for name, s in signal_sets.items():
    if name == 'F20 ICD (anchor)':
        continue
    only_in_this = s - f20_set
    source_contribution.append({
        'signal_source': name,
        'n_persons_in_signal': len(s),
        'n_persons_without_F20': len(only_in_this),
        'pct_of_no_F20_pool': 100 * len(only_in_this) / max(1, len(no_f20)),
    })

contrib_df = pd.DataFrame(source_contribution).sort_values('n_persons_without_F20', ascending=False)
print('How each signal source contributes to the "no-F20 but has psych signal" pool:')
display(contrib_df)


## 10. Save outputs

In [ ]:
# Save the candidate person list with provenance flags
candidates = pd.DataFrame({'person_id': sorted(no_f20)})
candidates['has_F20']                  = False  # by construction
candidates['has_visit_psych']          = candidates['person_id'].isin(signal_sets['visit_source_value psych'])
candidates['has_care_site_psych']      = candidates['person_id'].isin(signal_sets['care_site psych'])
candidates['has_procedure_psych']      = candidates['person_id'].isin(signal_sets['procedure psych'])
candidates['has_observation_referral'] = candidates['person_id'].isin(signal_sets['observation referral/psych'])
candidates['has_F22_F29']              = candidates['person_id'].isin(signal_sets['non-F20 psychosis (F22-F29)'])

out_path = OUTPUT_DIR / 'sch_referral_candidates_no_F20.parquet'
candidates.to_parquet(out_path, index=False)
print(f'Saved {len(candidates):,} candidate persons to {out_path}')

# Also save the signal-source contribution table for the writeup back to Yize
contrib_df.to_csv(OUTPUT_DIR / 'sch_referral_signal_contribution.csv', index=False)
print(f'Saved contribution table to {OUTPUT_DIR / "sch_referral_signal_contribution.csv"}')


## 11. Summary to send back to Yize

Take the numbers from cells 8, 9 above and report along these lines:

- "Out of `<N_F20>` persons with any F20 code, we found `<N_no_F20>` additional persons
  with mental-health service contact but no F20 ICD."
- "The largest contributor was `<top source>`, accounting for `<X>%` of the no-F20 pool."
- "If you want me to fold these in, candidate list is at `outputs/sch_referral_candidates_no_F20.parquet`
  with per-source provenance flags so you can pick which signals to trust."
- Open question: which combinations of signals are enough to count as a "schizophrenia
  candidate"? E.g., a single psychotherapy procedure is weak; F22-F29 ICD is strong;
  multiple psych visits at a psych care site over time is medium-to-strong.
